## Load Modules

In [39]:
import pandas as pd
import plotly.graph_objects as go
!pip install pvlib
from pvlib.pvsystem import PVSystem
from pvlib.location import Location
from pvlib.modelchain import ModelChain
from pvlib.temperature import TEMPERATURE_MODEL_PARAMETERS

## Set System Parameters

In [40]:
PV_Max_Power = 10000  # Maximum power in Watts
PV_Azimuth_deg = 180  # South-facing

## Load Data

In [41]:
# Load the dataset
url = 'https://raw.githubusercontent.com/OSU-fifem/OSU-ESE450/main/project/Yachats_Off_Grid_2023.csv'
df = pd.read_csv(url)

# Display column names so we can map them correctly if needed
print("Dataset Columns:", df.columns.tolist())

# Ensure everything is in UTC
df['Time'] = pd.to_datetime(df['Time'], utc=True)
df.set_index('Time', inplace=True)

display(df.head())

Dataset Columns: ['Time', 'Hour_of_Year', 'Hour_of_Day', 'Month', 'Temperature', 'Wind_Speed', 'Wind_Direction', 'DNI', 'DHI', 'GHI', 'Streamflow_m3_s', 'Base_Load_kW', 'Lighting_Plugs_kW', 'Induction_Range_kW', 'Total_Electrical_Load_kW', 'DHW_Demand_Liters']


,Hour_of_Year,Hour_of_Day,Month,Temperature,Wind_Speed,Wind_Direction,DNI,DHI,GHI,Streamflow_m3_s,Base_Load_kW,Lighting_Plugs_kW,Induction_Range_kW,Total_Electrical_Load_kW,DHW_Demand_Liters
Time,,,,,,,,,,,,,,,
2023-01-01 00:00:00+00:00,1,0,1,8.3,3.0,353,190,14,21,0.042,0.40,0.64,2.42,3.46,7.0
2023-01-01 01:00:00+00:00,2,1,1,8.1,3.0,352,0,0,0,0.039,0.36,0.56,2.09,3.00,5.1
2023-01-01 02:00:00+00:00,3,2,1,8.0,3.0,349,0,0,0,0.039,0.37,0.42,0.88,1.67,17.7
2023-01-01 03:00:00+00:00,4,3,1,7.9,2.9,349,0,0,0,0.029,0.40,0.24,0.00,0.63,0.0
2023-01-01 04:00:00+00:00,5,4,1,7.8,2.9,352,0,0,0,0.029,0.38,0.12,0.00,0.50,0.0


## Simulate PV

In [42]:
# Prepare the weather dataframe for pvlib.
# You may need to tweak these column name strings to match the exact headers in the CSV printed above.
weather = pd.DataFrame()
weather['ghi'] = df.get('GHI', df.get('ghi', 0)) # Global Horizontal Irradiance
weather['dni'] = df.get('DNI', df.get('dni', 0)) # Direct Normal Irradiance
weather['dhi'] = df.get('DHI', df.get('dhi', 0)) # Diffuse Horizontal Irradiance
weather['temp_air'] = df.get('Temperature', df.get('temp_air', 20)) # Dry Bulb Temperature
weather['wind_speed'] = df.get('Wind Speed', df.get('wind_speed', 0))

# 1. Define Location
latitude = 41.31054
longitude = -124.09559
tz = 'UTC'
site = Location(latitude, longitude, tz=tz)

# 2. Define System (10 kW peak)
# We assume the array is tilted at the latitude angle and faces South (azimuth=180)
system = PVSystem(
    surface_tilt=latitude,
    surface_azimuth=PV_Azimuth_deg,
    # module_parameters are for the simple "PVWatts" model
    # gamma_pdc is the temperature coefficient of power, which we set to a typical value for silicon panels
    module_parameters={'pdc0': PV_Max_Power, 'gamma_pdc': -0.004},
    inverter_parameters={'pdc0': PV_Max_Power},
    # Use the SAPM temperature model parameters for an open rack glass-glass configuration, which is common for residential PV systems
    temperature_model_parameters=TEMPERATURE_MODEL_PARAMETERS['sapm']['open_rack_glass_glass']
)

# 3. Create ModelChain - Don't specify dc_model.  Infer it from module_parameters.
mc = ModelChain(system, site, aoi_model='physical', spectral_model='no_loss')

# 4. Run Simulation
mc.run_model(weather)

# 5. Extract and Plot AC Power Output
ac_power = mc.results.ac

## Plot the Results

In [43]:
# Convert simulated AC power from Watts to kilowatts for comparison
pv_ac_power_kw = ac_power / 1000

fig = go.Figure()

# Add Solar Production Trace
fig.add_trace(go.Scatter(
    x=pv_ac_power_kw.index,
    y=pv_ac_power_kw,
    mode='lines',
    name='Solar Production (kW)',
    line=dict(color='orange')
))

# Add Total Electrical Load Trace
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['Total_Electrical_Load_kW'],
    mode='lines',
    name='Total Electrical Load (kW)',
    line=dict(color='blue')
))

# Update layout for better interactivity and labeling
fig.update_layout(
    title='Simulated 10kW Solar Array Production vs Total Electrical Load',
    xaxis_title='Time (UTC)',
    yaxis_title='Power (kW)',
    hovermode='x unified',
    template='plotly_white'
)

fig.show()

## Simulate System

In [44]:
# Calculate coincident PV production and curtailed PV using a for loop, directly populating the Series
coincident_pv_production = pd.Series(0.0, index=pv_ac_power_kw.index) # Initialize as Series
pv_curtailed = pd.Series(0.0, index=pv_ac_power_kw.index) # Initialize as Series

for i in range(len(pv_ac_power_kw)):
    pv_val = pv_ac_power_kw.iloc[i]
    load_val = df['Total_Electrical_Load_kW'].iloc[i]

    # Coincident PV production
    coincident_pv_production.iloc[i] = min(pv_val, load_val)

    # Curtailed PV production
    curtailed_val = pv_val - coincident_pv_production.iloc[i]
    pv_curtailed.iloc[i] = max(0.0, curtailed_val)

## Explore Simulated System Operation

In [45]:
fig = go.Figure()

# Add Solar Production Trace
fig.add_trace(go.Scatter(
    x=pv_ac_power_kw.index,
    y=pv_ac_power_kw,
    mode='lines',
    name='PV Generation (kW)',
    line=dict(color='orange')
))

# Add Total Electrical Load Trace
fig.add_trace(go.Scatter(
    x=df.index,
    y=df['Total_Electrical_Load_kW'],
    mode='lines',
    name='Electrical Load (kW)',
    line=dict(color='blue')
))

# Add Coincident PV Production Trace
fig.add_trace(go.Scatter(
    x=coincident_pv_production.index,
    y=coincident_pv_production,
    mode='lines',
    name='Coincident PV Production (kW)',
    line=dict(color='green')
))

# Update layout for better interactivity and labeling
fig.update_layout(
    title='PV System Simulation Results',
    xaxis_title='Time (UTC)',
    yaxis_title='Power (kW)',
    hovermode='x unified',
    template='plotly_white'
)

fig.show()

## Assess Performance

In [46]:
total_pv_kwh = pv_ac_power_kw.sum()
total_load_kwh = df['Total_Electrical_Load_kW'].sum()
total_satisfied_load_kwh = coincident_pv_production.sum()

# Calculate percentage of load satisfied based on coincident production
if total_load_kwh > 0:
    percentage_satisfied = (total_satisfied_load_kwh / total_load_kwh) * 100
else:
    percentage_satisfied = 0.0 # Handle case where total_load_kwh is zero to avoid division by zero

## Display Results

In [47]:
print(f"Max PV (kWh): {total_pv_kwh:.2f}")
print(f"Total Electrical Load (kWh): {total_load_kwh:.2f}")
print(f"Percentage of Load Satisfied (based on coincident PV) (%): {percentage_satisfied:.2f}")

Max PV (kWh): 14811.69
Total Electrical Load (kWh): 7807.53
Percentage of Load Satisfied (based on coincident PV) (%): 40.81
